In [9]:
import scipy.io.wavfile as wav
import numpy as np
from feature_extractor.webrtc_vad_filter import WebrtcVADFilter
from feature_extractor.python_speech_features import delta, rasta
# from feature_extractor.mfcc_feature_extractor import MfccFeatureExtractor
from feature_extractor.fir_filter import FIRFilter
import librosa
import os


#用于获取训练集与测试集中的音频特征,需要将音频进行6s分帧，重叠3s
class Data_features_extractor:
    
    BASE_MODE = "mfcc-base"
    DELTA_MODE = "mfcc-delta"
    DELTA_DELTA_MODE = "mfcc-delta-delta"
    RASTA_MODE = "raste"
    PLP_MODE = "plp"
    RASTA_PLP_MODE = "raste-plp"
    MFCC_RASTA_MODE = "mfcc-raste"
    
    MODES = [BASE_MODE, DELTA_MODE, DELTA_DELTA_MODE]
    
    
    def __init__(self, trainRtest,extract_mode="mfcc-delta-delta"):
        self.extract_mode=extract_mode    
        self.datas=[]
        self.win_length = 0.03
        self.cur_path=os.path.join(os.getcwd(),'split_data',trainRtest)

        #获取其中类别，AD,HC,MCI
        self.Classes=os.listdir(self.cur_path)
        count=-1
        
        for Class in self.Classes:
            count+=1
            self.datas.append([])
            files=os.listdir(os.path.join(self.cur_path,Class))
            for file in files:
                # use webrtc VAD for filtering:
                rate, signal = wav.read(os.path.join(self.cur_path,Class,file))
                original_len = len(signal)
                print(Class,"File %s audio file result: %d/%d" % (file, len(signal), original_len))
                split_array=self.split_accu(signal,len(signal))
                for split in split_array:
                    self.datas[count].append(self.extract(split,rate))
                print('Extracted file: {}'.format(file))

                
    #音频重叠切片，只取前30s，帧长6s，重叠4s
    def split_accu(self,arr,length):
        if len(arr)<length:
            zeros = [0] * (480000-len(arr))
            arr+=zeros
        new_arr=[]
        start=0
        end=96000
        while end<=480000:
            new_arr.append(arr[start:end])
            start+=32000
            end=start+96000
        return new_arr

    
    #读取音频数据，将其转换为对应特征谱
    def extract(self, signal, rate):
        #if self.extract_mode == self.RASTA_MODE || self.extract_mode == self.RASTA_PLP_MODE:
        #rasta_feat = np.transpose(rasta.rastaplp(signal, fs=rate, modelorder=12))
        
        #if self.extract_mode == self.DELTA_MODE || selfextract_mode == self.DELTA_MODE || self.extract_mode == self.BASE_MODE:
        mfcc_feat = np.transpose(rasta.melfcc(signal, fs=rate))
        #plp_feat = np.transpose(rasta.melfcc(signal, fs=rate, fbtype='bark'))

        if self.extract_mode == self.DELTA_MODE:
            mfcc_feat = np.concatenate((mfcc_feat, delta(mfcc_feat, 2)), axis=1)
        elif self.extract_mode == self.DELTA_DELTA_MODE:
            delta_feat = delta(mfcc_feat, 2)
            mfcc_feat = np.concatenate((mfcc_feat, delta_feat, delta(delta_feat, 2)), axis=1)
        
        return mfcc_feat
        #feature_vector = np.concatenate((rasta_feat, mfcc_feat, plp_feat), axis=1)  # mfcc + rasta + plp
        # feature_vector = np.concatenate((mfcc_feat, plp_feat), axis=1)            # mfcc + plp
        # feature_vector = rasta_feat                                               # rasta
        # feature_vector = plp_feat                                                 # plp
        # feature_vector = mfcc_feat                                                # mfcc

#         if self.feature_vectors.size == 0:
#             self.feature_vectors = np.copy(feature_vector)
#         else:
#             self.feature_vectors = np.concatenate((self.feature_vectors, feature_vector))

        # return feature_vector
  

    def get_data(self):
        return np.array(self.datas)
        

In [10]:
#datas=Data_features_extractor('test').get_data()
#print(datas.shape)

AD File 101.wav audio file result: 767840/767840
Extracted file: 101.wav
AD File 104.wav audio file result: 767840/767840
Extracted file: 104.wav
AD File 105.wav audio file result: 767840/767840
Extracted file: 105.wav
AD File 108.wav audio file result: 767840/767840
Extracted file: 108.wav
AD File 11.wav audio file result: 767840/767840
Extracted file: 11.wav
AD File 115.wav audio file result: 767840/767840
Extracted file: 115.wav
AD File 117.wav audio file result: 767840/767840
Extracted file: 117.wav
AD File 118.wav audio file result: 767840/767840
Extracted file: 118.wav
AD File 12.wav audio file result: 767840/767840
Extracted file: 12.wav
AD File 13.wav audio file result: 767840/767840
Extracted file: 13.wav
AD File 15.wav audio file result: 767840/767840
Extracted file: 15.wav
AD File 17.wav audio file result: 767840/767840
Extracted file: 17.wav
AD File 2.wav audio file result: 767840/767840
Extracted file: 2.wav
AD File 20.wav audio file result: 767840/767840
Extracted file: 2

MCI File 80.wav audio file result: 767840/767840
Extracted file: 80.wav
MCI File 83.wav audio file result: 767840/767840
Extracted file: 83.wav
MCI File 90.wav audio file result: 767840/767840
Extracted file: 90.wav
MCI File 91.wav audio file result: 767840/767840
Extracted file: 91.wav
(3,)


C:\Users\dell\AppData\Local\Temp\ipykernel_12888\48629851.py:96: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(self.datas)


In [11]:
#for List in datas:
    #print(np.array(List).shape)

(455, 297, 39)
(585, 297, 39)
(507, 297, 39)
